# StockSaathi: Sample Store Data and Tool Testing
### Aakash Goswami (065061) | FORE School of Management | AI Application End Term Project

This notebook creates the sample data that powers **StockSaathi**, an AI stock assistant for a small Indian retailer (use case 11: inventory and stock query assistant), and then tests every tool the chatbot can call.

**Why synthetic data?** Real store data is private. A carefully simulated dataset lets me control and explain every scenario the demo needs: fast movers about to run out, products already out of stock, near-expiry dairy and bread, slow-moving dead stock, and seasonal trends.

**Outputs (saved to `data/`):**

| File | Contents |
|---|---|
| `distributors.csv` | 7 distributors with visit day, visit frequency and lead time |
| `skus.csv` | 74 FMCG products with brand, category, MRP, cost, case size, current stock, earliest expiry and Hindi/Hinglish aliases |
| `sales_daily.csv` | 90 days of daily unit sales per product |


## 1. Setup

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

rng = np.random.default_rng(42)          # fixed seed: the same data every run
AS_OF = pd.Timestamp("2026-10-03")       # the store's 'today' for the demo
DAYS = 90
DATA = Path("data"); DATA.mkdir(exist_ok=True)
print("Data window:", (AS_OF - pd.Timedelta(days=DAYS-1)).date(), "to", AS_OF.date())

Data window: 2026-07-06 to 2026-10-03


## 2. Store and distributors

The fictional store is **Sharma General Store, Lajpat Nagar, New Delhi**. Each distributor's salesman visits on a fixed weekday (the usual beat plan in FMCG general trade). Dairy is delivered daily.

In [2]:
distributors = pd.DataFrame([
    # distributor,                    visit_day,   visit_every_days, lead_time_days
    ["Gupta Agencies (HUL)",          "Monday",    7, 2],
    ["Shree Balaji Traders (ITC)",    "Tuesday",   7, 2],
    ["Nestle CFA, Okhla",             "Wednesday", 7, 3],
    ["Dabur Super Stockist",          "Thursday",  7, 3],
    ["Kapoor Distributors (Britannia, Parle)", "Friday", 7, 2],
    ["Mehta Trading Co (Tata, Marico, Colgate)", "Saturday", 7, 3],
    ["Amul Milk Depot",               "Daily",     1, 1],
], columns=["distributor", "visit_day", "visit_every_days", "lead_time_days"])
distributors

,distributor,visit_day,visit_every_days,lead_time_days
0,Gupta Agencies (HUL),Monday,7,2
1,Shree Balaji Traders (ITC),Tuesday,7,2
2,"Nestle CFA, Okhla",Wednesday,7,3
3,Dabur Super Stockist,Thursday,7,3
4,"Kapoor Distributors (Britannia, Parle)",Friday,7,2
5,"Mehta Trading Co (Tata, Marico, Colgate)",Saturday,7,3
6,Amul Milk Depot,Daily,1,1


## 3. Product catalogue

Each row: brand, product, category, unit, MRP (Rs), case size, typical daily sales, shelf life in days, and aliases. Aliases include Hindi (Devanagari) and Hinglish names so the chatbot can match how a shop owner actually talks, for example "मैगी" or "chai patti".

In [3]:
HUL, ITC, NES, DAB = distributors.distributor[:4]
BRI, TMC, AMUL = distributors.distributor[4], distributors.distributor[5], distributors.distributor[6]

catalogue = [
 # brand, product, category, unit, mrp, case, daily, shelf_days, distributor, aliases
 ("Dabur", "Dabur Red Paste 100g", "Personal care", "tubes", 62, 12, 2.2, 540, DAB, "red paste|dabur red|toothpaste|दंत मंजन|डाबर रेड"),
 ("Dabur", "Dabur Red Paste 200g", "Personal care", "tubes", 112, 12, 1.2, 540, DAB, "red paste|dabur red|toothpaste"),
 ("Dabur", "Dabur Honey 250g", "Health", "jars", 125, 12, 0.8, 540, DAB, "honey|shahad|शहद|madhu"),
 ("Dabur", "Dabur Honey 500g", "Health", "jars", 225, 6, 0.5, 540, DAB, "honey|shahad|शहद|madhu"),
 ("Dabur", "Dabur Chyawanprash 500g", "Health", "jars", 245, 6, 0.6, 540, DAB, "chyawanprash|च्यवनप्राश|chavanprash"),
 ("Dabur", "Real Fruit Juice Mixed Fruit 1L", "Beverages", "packs", 120, 12, 0.9, 180, DAB, "real juice|mixed fruit juice|juice|जूस"),
 ("Dabur", "Real Orange Juice 1L", "Beverages", "packs", 120, 12, 0.6, 180, DAB, "real orange|orange juice|juice|जूस"),
 ("Dabur", "Dabur Amla Hair Oil 180ml", "Personal care", "bottles", 110, 12, 1.0, 720, DAB, "amla oil|amla tel|hair oil|आंवला तेल"),
 ("Dabur", "Vatika Shampoo 180ml", "Personal care", "bottles", 155, 12, 0.5, 720, DAB, "vatika|shampoo"),
 ("Dabur", "Hajmola 120 Tablets", "Health", "bottles", 60, 24, 1.3, 540, DAB, "hajmola|हाजमोला|churan goli"),
 ("Dabur", "Odonil Air Freshener 50g", "Home care", "pieces", 55, 24, 0.7, 720, DAB, "odonil|air freshener|bathroom freshener"),
 ("Dabur", "Odomos Cream 50g", "Personal care", "tubes", 95, 24, 0.15, 720, DAB, "odomos|mosquito cream|machhar cream"),
 ("ITC", "Aashirvaad Atta 5kg", "Staples", "bags", 285, 4, 3.0, 120, ITC, "atta|aata|आटा|aashirvaad|gehun atta"),
 ("ITC", "Aashirvaad Atta 10kg", "Staples", "bags", 545, 2, 1.0, 120, ITC, "atta|aata|आटा|aashirvaad"),
 ("ITC", "Sunfeast Dark Fantasy 75g", "Biscuits", "packs", 40, 24, 2.5, 240, ITC, "dark fantasy|choco fills"),
 ("ITC", "Sunfeast Marie Light 250g", "Biscuits", "packs", 40, 24, 1.5, 240, ITC, "marie light|sunfeast marie"),
 ("ITC", "Bingo Mad Angles 66g", "Snacks", "packs", 20, 48, 4.0, 150, ITC, "mad angles|bingo"),
 ("ITC", "Bingo Tedhe Medhe 90g", "Snacks", "packs", 20, 48, 3.0, 150, ITC, "tedhe medhe|bingo"),
 ("ITC", "YiPPee Noodles 4-pack", "Noodles", "packs", 60, 12, 1.0, 270, ITC, "yippee|yippee noodles"),
 ("ITC", "Classmate Notebook 172 pages", "Stationery", "pieces", 60, 24, 0.8, 3650, ITC, "classmate|notebook|copy|कॉपी"),
 ("ITC", "Fiama Shower Gel 250ml", "Personal care", "bottles", 199, 12, 0.1, 720, ITC, "fiama|shower gel|body wash"),
 ("ITC", "Savlon Handwash 200ml", "Personal care", "bottles", 99, 24, 0.7, 720, ITC, "savlon|handwash"),
 ("ITC", "Mangaldeep Agarbatti", "Pooja", "packs", 50, 24, 1.2, 1095, ITC, "agarbatti|अगरबत्ती|mangaldeep|dhoop"),
 ("HUL", "Surf Excel Easy Wash 1kg", "Home care", "packs", 140, 12, 1.8, 720, HUL, "surf|surf excel|washing powder|सर्फ"),
 ("HUL", "Rin Detergent Bar 250g", "Home care", "bars", 35, 40, 2.0, 720, HUL, "rin|rin bar|kapde ka sabun"),
 ("HUL", "Vim Bar 300g", "Home care", "bars", 35, 40, 2.2, 720, HUL, "vim|bartan sabun|विम"),
 ("HUL", "Lifebuoy Soap 4x100g", "Personal care", "packs", 148, 12, 1.0, 720, HUL, "lifebuoy|soap|sabun|साबुन"),
 ("HUL", "Lux Soap 4x100g", "Personal care", "packs", 160, 12, 0.8, 720, HUL, "lux|soap|sabun|साबुन"),
 ("HUL", "Dove Shampoo 180ml", "Personal care", "bottles", 220, 12, 0.4, 720, HUL, "dove|shampoo"),
 ("HUL", "Clinic Plus Shampoo 175ml", "Personal care", "bottles", 110, 12, 0.7, 720, HUL, "clinic plus|shampoo"),
 ("HUL", "Brooke Bond Red Label Tea 500g", "Tea and coffee", "packs", 290, 12, 1.4, 540, HUL, "red label|chai patti|chai|चाय|चायपत्ती|tea"),
 ("HUL", "Taaza Tea 250g", "Tea and coffee", "packs", 125, 24, 1.1, 540, HUL, "taaza|chai patti|chai|चाय|tea"),
 ("HUL", "Kissan Fresh Tomato Ketchup 950g", "Sauces", "bottles", 150, 12, 0.6, 365, HUL, "kissan|ketchup|sauce|सॉस"),
 ("HUL", "Knorr Classic Tomato Soup", "Sauces", "packs", 55, 24, 0.5, 365, HUL, "knorr|soup"),
 ("HUL", "Closeup Toothpaste 150g", "Personal care", "tubes", 110, 12, 0.9, 720, HUL, "closeup|close up|toothpaste"),
 ("HUL", "Pepsodent Toothpaste 150g", "Personal care", "tubes", 95, 12, 0.7, 720, HUL, "pepsodent|toothpaste"),
 ("HUL", "Horlicks Classic Malt 500g", "Health", "jars", 265, 6, 0.5, 365, HUL, "horlicks|हॉर्लिक्स"),
 ("Nestle", "Maggi 2-Minute Noodles 70g", "Noodles", "packs", 14, 96, 18.0, 270, NES, "maggi|मैगी|noodles|maggie|single maggi"),
 ("Nestle", "Maggi 2-Minute Noodles 4-pack 280g", "Noodles", "packs", 56, 24, 3.0, 270, NES, "maggi|मैगी|maggi 4 pack|maggi family pack|maggie"),
 ("Nestle", "Maggi Masala-ae-Magic 6g", "Sauces", "sachets", 5, 120, 6.0, 365, NES, "masala ae magic|maggi masala"),
 ("Nestle", "Nescafe Classic 50g", "Tea and coffee", "jars", 190, 12, 0.6, 540, NES, "nescafe|coffee|कॉफी"),
 ("Nestle", "KitKat 4 Finger 37g", "Confectionery", "bars", 30, 48, 3.5, 270, NES, "kitkat|kit kat|chocolate"),
 ("Nestle", "Munch 23g", "Confectionery", "bars", 10, 96, 5.0, 270, NES, "munch|chocolate"),
 ("Nestle", "Everyday Dairy Whitener 400g", "Dairy", "packs", 225, 12, 0.7, 365, NES, "everyday|milk powder|dairy whitener"),
 ("Nestle", "Cerelac Wheat 300g", "Health", "packs", 275, 12, 0.0, 365, NES, "cerelac|baby food"),
 ("Nestle", "Maggi Hot and Sweet Sauce 500g", "Sauces", "bottles", 125, 12, 0.4, 365, NES, "maggi sauce|hot and sweet|ketchup|sauce"),
 ("Britannia", "Britannia Good Day Cashew 200g", "Biscuits", "packs", 40, 36, 2.5, 240, BRI, "good day|गुड डे"),
 ("Britannia", "Britannia Marie Gold 250g", "Biscuits", "packs", 35, 36, 2.0, 240, BRI, "marie gold|marie"),
 ("Britannia", "Britannia Bourbon 150g", "Biscuits", "packs", 35, 36, 1.8, 240, BRI, "bourbon"),
 ("Britannia", "Britannia White Bread 400g", "Bakery", "loaves", 45, 12, 6.0, 5, BRI, "bread|ब्रेड|double roti|white bread"),
 ("Britannia", "Britannia Cake Fruity Fun 60g", "Bakery", "packs", 20, 48, 2.0, 120, BRI, "cake|fruity fun"),
 ("Britannia", "Britannia 50-50 Maska Chaska", "Biscuits", "packs", 30, 36, 1.5, 240, BRI, "50 50|maska chaska|fifty fifty"),
 ("Parle", "Parle-G 250g", "Biscuits", "packs", 25, 48, 12.0, 240, BRI, "parle g|parle-g|parleg|पारले जी|glucose biscuit"),
 ("Parle", "Parle-G 800g", "Biscuits", "packs", 80, 12, 1.5, 240, BRI, "parle g|parle-g|parleg|पारले जी"),
 ("Parle", "Parle Hide and Seek 120g", "Biscuits", "packs", 40, 36, 1.6, 240, BRI, "hide and seek|hide n seek"),
 ("Parle", "Parle Krackjack 200g", "Biscuits", "packs", 35, 36, 1.4, 240, BRI, "krackjack"),
 ("Parle", "Parle Monaco 120g", "Biscuits", "packs", 30, 36, 1.5, 240, BRI, "monaco"),
 ("Amul", "Amul Taaza Toned Milk 500ml", "Dairy", "pouches", 28, 24, 30.0, 2, AMUL, "milk|doodh|दूध|amul taaza|taaza milk"),
 ("Amul", "Amul Gold Full Cream Milk 500ml", "Dairy", "pouches", 34, 24, 18.0, 2, AMUL, "milk|doodh|दूध|amul gold|full cream"),
 ("Amul", "Amul Butter 100g", "Dairy", "packs", 58, 20, 4.0, 180, AMUL, "butter|makhan|मक्खन|amul butter"),
 ("Amul", "Amul Cheese Slices 200g", "Dairy", "packs", 140, 12, 0.8, 180, AMUL, "cheese|cheese slice|चीज़"),
 ("Amul", "Amul Masti Dahi 400g", "Dairy", "cups", 35, 12, 5.0, 10, AMUL, "dahi|curd|दही"),
 ("Amul", "Amul Malai Paneer 200g", "Dairy", "packs", 95, 12, 2.5, 7, AMUL, "paneer|पनीर"),
 ("Tata", "Tata Salt 1kg", "Staples", "packs", 28, 30, 5.0, 720, TMC, "salt|namak|नमक|tata namak"),
 ("Tata", "Tata Tea Premium 500g", "Tea and coffee", "packs", 285, 12, 1.2, 540, TMC, "tata tea|chai patti|chai|चाय|tea"),
 ("Tata", "Tata Tea Gold 250g", "Tea and coffee", "packs", 175, 24, 0.8, 540, TMC, "tata tea gold|chai patti|chai|tea"),
 ("Tata", "Tata Sampann Toor Dal 1kg", "Staples", "packs", 180, 10, 1.5, 270, TMC, "toor dal|arhar dal|dal|दाल|तूर दाल"),
 ("Marico", "Parachute Coconut Oil 200ml", "Personal care", "bottles", 100, 24, 1.6, 540, TMC, "parachute|coconut oil|nariyal tel|नारियल तेल"),
 ("Marico", "Saffola Gold Oil 1L", "Staples", "bottles", 210, 12, 1.2, 270, TMC, "saffola|cooking oil|tel|तेल|refined"),
 ("Marico", "Saffola Masala Oats 400g", "Breakfast", "packs", 195, 12, 0.5, 270, TMC, "oats|saffola oats"),
 ("Marico", "Nihar Shanti Amla Oil 200ml", "Personal care", "bottles", 75, 24, 0.9, 540, TMC, "nihar|amla oil|hair oil"),
 ("Colgate", "Colgate Strong Teeth 200g", "Personal care", "tubes", 120, 12, 1.8, 720, TMC, "colgate|toothpaste|कोलगेट"),
 ("Colgate", "Colgate MaxFresh 150g", "Personal care", "tubes", 110, 12, 0.9, 720, TMC, "maxfresh|colgate|toothpaste"),
 ("Colgate", "Palmolive Soap 3x75g", "Personal care", "packs", 110, 12, 0.6, 720, TMC, "palmolive|soap|sabun"),
]
cols = ["brand", "product", "category", "unit", "mrp", "case_size", "base_daily",
        "shelf_days", "distributor", "aliases"]
skus = pd.DataFrame(catalogue, columns=cols)
skus.insert(0, "sku_id", [f"SKU{i:03d}" for i in range(1, len(skus) + 1)])
skus["cost_price"] = (skus["mrp"] * rng.uniform(0.80, 0.88, len(skus))).round(2)  # retailer margin 12 to 20%
print(len(skus), "products across", skus.brand.nunique(), "brands and", skus.category.nunique(), "categories")
skus.groupby("brand").size()

74 products across 10 brands and 16 categories


brand
Amul          6
Britannia     6
Colgate       3
Dabur        12
HUL          14
ITC          11
Marico        4
Nestle        9
Parle         5
Tata          4
dtype: int64

## 4. Daily sales simulation

Units sold per day follow a Poisson distribution around each product's typical rate, adjusted for:

* **Weekends:** Saturday and Sunday sell about 25% more.
* **Salary week:** the first 7 days of each month sell about 15% more.
* **Seasonality:** fruit juice and mosquito cream fade as summer ends; Chyawanprash and Horlicks rise as winter approaches.
* **Cerelac** has no sales in the last 28 days (to create a dead-stock case).

In [4]:
dates = pd.date_range(end=AS_OF, periods=DAYS, freq="D")
t = np.linspace(0, 1, DAYS)                      # 0 = start of window, 1 = today
weekend = np.where(dates.dayofweek >= 5, 1.25, 1.0)
salary = np.where(dates.day <= 7, 1.15, 1.0)

trend = {   # multiplier path from start to end of the 90 days
    "Real Fruit Juice Mixed Fruit 1L": (1.8, 0.6),
    "Real Orange Juice 1L": (1.8, 0.6),
    "Odomos Cream 50g": (2.5, 0.8),
    "Dabur Chyawanprash 500g": (0.2, 3.0),
    "Horlicks Classic Malt 500g": (0.7, 1.4),
}
rows = []
for _, s in skus.iterrows():
    a, b = trend.get(s["product"], (1.0, 1.0))
    seasonal = a + (b - a) * t
    lam = s["base_daily"] * weekend * salary * seasonal
    if s["product"] == "Cerelac Wheat 300g":
        lam = np.where(dates > AS_OF - pd.Timedelta(days=28), 0, 0.4)
    units = rng.poisson(lam)
    rows.append(pd.DataFrame({"date": dates, "sku_id": s["sku_id"], "units": units}))
sales = pd.concat(rows, ignore_index=True)
print(f"{len(sales):,} sales rows, {sales.units.sum():,} units sold in 90 days")
sales.groupby("date").units.sum().tail(14)

6,660 sales rows, 19,032 units sold in 90 days


date
2026-09-20    231
2026-09-21    186
2026-09-22    184
2026-09-23    184
2026-09-24    185
2026-09-25    188
2026-09-26    224
2026-09-27    241
2026-09-28    176
2026-09-29    178
2026-09-30    198
2026-10-01    224
2026-10-02    229
2026-10-03    291
Name: units, dtype: int64

## 5. Current stock and expiry scenarios

Most products get a healthy 8 to 25 days of stock. A few are set deliberately so the demo has something to find:

| Scenario | Products |
|---|---|
| Out of stock | Maggi 4-pack, Dark Fantasy |
| About to run out | Maggi 70g, Parle-G 250g, Amul Butter, Surf Excel, Dabur Red 100g |
| Overstock / dead stock | Cerelac, Fiama Shower Gel, Odomos |
| Near expiry | Bread, Dahi, Paneer, Real Mixed Fruit juice, Cerelac |
| Rising demand (tests the reorder logic) | Chyawanprash with only a little stock |

In [5]:
recent_rate = (sales[sales.date > AS_OF - pd.Timedelta(days=28)]
               .groupby("sku_id").units.mean())
skus["current_stock"] = [max(0, int(round(recent_rate[s] * rng.uniform(8, 25))))
                         for s in skus.sku_id]
skus.loc[skus["product"].str.contains("Amul Taaza|Amul Gold"), "current_stock"] = [42, 26]

set_stock = {
    "Maggi 2-Minute Noodles 4-pack 280g": 0, "Sunfeast Dark Fantasy 75g": 0,
    "Maggi 2-Minute Noodles 70g": 22, "Parle-G 250g": 14, "Amul Butter 100g": 3,
    "Surf Excel Easy Wash 1kg": 2, "Dabur Red Paste 100g": 4,
    "Cerelac Wheat 300g": 14, "Fiama Shower Gel 250ml": 18, "Odomos Cream 50g": 30,
    "Dabur Chyawanprash 500g": 5, "Real Fruit Juice Mixed Fruit 1L": 24,
}
for p, q in set_stock.items():
    skus.loc[skus["product"] == p, "current_stock"] = q

# earliest expiry of the oldest batch on the shelf, and how many units are in that batch
days_left = [int(rng.integers(max(2, sd // 4), max(3, sd // 2))) for sd in skus.shelf_days]
skus["earliest_expiry"] = [AS_OF + pd.Timedelta(days=d) for d in days_left]
skus["expiry_batch_qty"] = (skus["current_stock"] * rng.uniform(0.3, 0.6, len(skus))).round().astype(int)
set_expiry = {   # product: (days to expiry, units in that batch)
    "Britannia White Bread 400g": (1, 14), "Amul Masti Dahi 400g": (3, 25),
    "Amul Malai Paneer 200g": (2, 9), "Real Fruit Juice Mixed Fruit 1L": (21, 24),
    "Cerelac Wheat 300g": (20, 14), "KitKat 4 Finger 37g": (45, 30),
}
for p, (d, q) in set_expiry.items():
    i = skus.index[skus["product"] == p]
    skus.loc[i, "earliest_expiry"] = AS_OF + pd.Timedelta(days=d)
    skus.loc[i, "expiry_batch_qty"] = q
skus["expiry_batch_qty"] = np.minimum(skus["expiry_batch_qty"], skus["current_stock"])

out_cols = ["sku_id", "brand", "product", "category", "unit", "mrp", "cost_price", "case_size",
            "distributor", "current_stock", "earliest_expiry", "expiry_batch_qty", "aliases"]
skus[out_cols].to_csv(DATA / "skus.csv", index=False)
sales.to_csv(DATA / "sales_daily.csv", index=False)
distributors.to_csv(DATA / "distributors.csv", index=False)
print("Saved:", [f.name for f in DATA.iterdir()])
skus[out_cols].head(8)

Saved: ['skus.csv', 'distributors.csv', 'sales_daily.csv']


,sku_id,brand,product,category,unit,mrp,cost_price,case_size,distributor,current_stock,earliest_expiry,expiry_batch_qty,aliases
0,SKU001,Dabur,Dabur Red Paste 100g,Personal care,tubes,62,53.44,12,Dabur Super Stockist,4,2027-05-27,2,red paste|dabur red|toothpaste|दंत मंजन|डाबर रेड
1,SKU002,Dabur,Dabur Red Paste 200g,Personal care,tubes,112,93.53,12,Dabur Super Stockist,14,2027-02-24,7,red paste|dabur red|toothpaste
2,SKU003,Dabur,Dabur Honey 250g,Health,jars,125,108.59,12,Dabur Super Stockist,10,2027-03-09,4,honey|shahad|शहद|madhu
3,SKU004,Dabur,Dabur Honey 500g,Health,jars,225,192.55,6,Dabur Super Stockist,17,2027-03-28,9,honey|shahad|शहद|madhu
4,SKU005,Dabur,Dabur Chyawanprash 500g,Health,jars,245,197.85,6,Dabur Super Stockist,5,2027-06-03,2,chyawanprash|च्यवनप्राश|chavanprash
5,SKU006,Dabur,Real Fruit Juice Mixed Fruit 1L,Beverages,packs,120,105.37,12,Dabur Super Stockist,24,2026-10-24,24,real juice|mixed fruit juice|juice|जूस
6,SKU007,Dabur,Real Orange Juice 1L,Beverages,packs,120,103.31,12,Dabur Super Stockist,6,2026-12-18,3,real orange|orange juice|juice|जूस
7,SKU008,Dabur,Dabur Amla Hair Oil 180ml,Personal care,bottles,110,94.92,12,Dabur Super Stockist,22,2027-07-29,7,amla oil|amla tel|hair oil|आंवला तेल


## 6. Test the tools the chatbot will call

The app imports `inventory_tools.py`. Below, every tool is tested directly, without any AI, to confirm the numbers are right before Gemini is allowed to use them.

**Reorder logic (periodic review, the standard for weekly distributor visits):**

* Average daily sales and its variability come from the last 28 days.
* Safety stock = 1.65 x daily standard deviation x square root of lead time (about a 95% service level).
* Order up to = average daily sales x (lead time + days until the next visit) + safety stock.
* Suggested order = order-up-to level minus current stock, rounded up to full cases.
* **Critical** = stock will run out before a new order can arrive; **Low** = at or below the reorder point.

In [6]:
import importlib, json
import inventory_tools as it
importlib.reload(it)

skus_l, sales_l, dist_l = it.load_data("data")
m = it.compute_metrics(skus_l, sales_l, dist_l)
print("As of:", m.attrs["as_of"])
m["status"].value_counts()

As of: 2026-10-03


status
OK                     62
Critical                6
Overstock               2
Out of stock            2
No sales in 28 days     1
Low                     1
Name: count, dtype: int64

In [7]:
def show(x): print(json.dumps(x, indent=1, ensure_ascii=False)[:1800])

print("--- check_stock('Maggi kitni bachi hai') ---"); show(it.check_stock(m, "Maggi kitni bachi hai"))

--- check_stock('Maggi kitni bachi hai') ---
{
 "found": true,
 "needs_clarification": true,
 "query": "Maggi kitni bachi hai",
 "message": "Several products match. Ask which one the owner means.",
 "options": [
  {
   "product": "Maggi 2-Minute Noodles 4-pack 280g",
   "current_stock": 0
  },
  {
   "product": "Maggi 2-Minute Noodles 70g",
   "current_stock": 22
  }
 ]
}


In [8]:
for q in ["मैगी", "Parle G", "parleg", "doodh", "Dabur", "chai patti", "Colgte", "iPhone"]:
    kind, hits = it.find_products(m, q)
    print(f"{q!r:16} -> {kind:10} {hits['product'].head(4).tolist()}")

'मैगी'           -> ambiguous  ['Maggi 2-Minute Noodles 4-pack 280g', 'Maggi 2-Minute Noodles 70g']


'Parle G'        -> ambiguous  ['Parle-G 250g', 'Parle-G 800g']
'parleg'         -> ambiguous  ['Parle-G 250g', 'Parle-G 800g']


'doodh'          -> ambiguous  ['Amul Taaza Toned Milk 500ml', 'Amul Gold Full Cream Milk 500ml']
'Dabur'          -> brand      ['Dabur Red Paste 100g', 'Dabur Red Paste 200g', 'Dabur Honey 250g', 'Dabur Honey 500g']


'chai patti'     -> ambiguous  ['Taaza Tea 250g', 'Brooke Bond Red Label Tea 500g', 'Tata Tea Gold 250g', 'Tata Tea Premium 500g']
'Colgte'         -> ambiguous  ['Palmolive Soap 3x75g', 'Colgate MaxFresh 150g', 'Colgate Strong Teeth 200g']


'iPhone'         -> partial    ['Dabur Honey 500g', 'Dabur Honey 250g']


In [9]:
print("--- low_stock() ---"); show(it.low_stock(m, ""))

--- low_stock() ---
{
 "count": 9,
 "items": [
  {
   "product": "Sunfeast Dark Fantasy 75g",
   "status": "Out of stock",
   "current_stock": 0,
   "days_of_cover": 0.0,
   "suggested_order_qty": 48,
   "distributor": "Shree Balaji Traders (ITC)",
   "visit_day": "Tuesday"
  },
  {
   "product": "Maggi 2-Minute Noodles 4-pack 280g",
   "status": "Out of stock",
   "current_stock": 0,
   "days_of_cover": 0.0,
   "suggested_order_qty": 48,
   "distributor": "Nestle CFA, Okhla",
   "visit_day": "Wednesday"
  },
  {
   "product": "Amul Butter 100g",
   "status": "Critical",
   "current_stock": 3,
   "days_of_cover": 0.6,
   "suggested_order_qty": 20,
   "distributor": "Amul Milk Depot",
   "visit_day": "Daily"
  },
  {
   "product": "Surf Excel Easy Wash 1kg",
   "status": "Critical",
   "current_stock": 2,
   "days_of_cover": 1.0,
   "suggested_order_qty": 24,
   "distributor": "Gupta Agencies (HUL)",
   "visit_day": "Monday"
  },
  {
   "product": "Parle-G 250g",
   "status": "Critical"

In [10]:
print("--- expiring_soon(7) ---"); show(it.expiring_soon(m, 7))
print("--- expiring_soon(-5) (bad input is corrected) ---"); show(it.expiring_soon(m, -5))

--- expiring_soon(7) ---
{
 "within_days": 7,
 "count": 5,
 "total_value_at_risk_rs": 761.0,
 "items": [
  {
   "product": "Britannia White Bread 400g",
   "expiry_date": "04 Oct 2026",
   "days_left": 1,
   "units_in_batch": 14,
   "units_likely_unsold": 7,
   "value_at_risk_rs": 256.0
  },
  {
   "product": "Amul Taaza Toned Milk 500ml",
   "expiry_date": "05 Oct 2026",
   "days_left": 2,
   "units_in_batch": 15,
   "units_likely_unsold": 0,
   "value_at_risk_rs": 0.0
  },
  {
   "product": "Amul Gold Full Cream Milk 500ml",
   "expiry_date": "05 Oct 2026",
   "days_left": 2,
   "units_in_batch": 9,
   "units_likely_unsold": 0,
   "value_at_risk_rs": 0.0
  },
  {
   "product": "Amul Malai Paneer 200g",
   "expiry_date": "05 Oct 2026",
   "days_left": 2,
   "units_in_batch": 9,
   "units_likely_unsold": 3,
   "value_at_risk_rs": 241.0
  },
  {
   "product": "Amul Masti Dahi 400g",
   "expiry_date": "06 Oct 2026",
   "days_left": 3,
   "units_in_batch": 25,
   "units_likely_unsold": 9,

In [11]:
print("--- purchase_order('Nestle') ---"); show(it.purchase_order(m, "Nestle"))

--- purchase_order('Nestle') ---
{
 "orders": [
  {
   "distributor": "Nestle CFA, Okhla",
   "visit_day": "Wednesday",
   "lines": [
    {
     "product": "Maggi 2-Minute Noodles 4-pack 280g",
     "order_qty": 48,
     "case_size": 24,
     "est_cost_rs": 2199.0
    },
    {
     "product": "Maggi 2-Minute Noodles 70g",
     "order_qty": 192,
     "case_size": 96,
     "est_cost_rs": 2252.0
    }
   ],
   "total_rs": 4451.0
  }
 ],
 "grand_total_rs": 4451.0,
 "note": "Draft only. Quantities top stock up to cover lead time + time to next visit + safety stock, rounded up to full cases."
}


In [12]:
print("--- sales_report(7) ---"); show(it.sales_report(m, sales_l, 7, ""))
print("--- dead_stock() ---"); show(it.dead_stock(m))

--- sales_report(7) ---
{
 "period_days": 7,
 "total_units": 1537,
 "total_revenue_rs": 93964.0,
 "top_sellers_by_revenue": [
  {
   "product": "Aashirvaad Atta 5kg",
   "units": 37,
   "revenue_rs": 10545.0
  },
  {
   "product": "Amul Taaza Toned Milk 500ml",
   "units": 230,
   "revenue_rs": 6440.0
  },
  {
   "product": "Amul Gold Full Cream Milk 500ml",
   "units": 135,
   "revenue_rs": 4590.0
  },
  {
   "product": "Brooke Bond Red Label Tea 500g",
   "units": 15,
   "revenue_rs": 4350.0
  },
  {
   "product": "Dabur Chyawanprash 500g",
   "units": 17,
   "revenue_rs": 4165.0
  }
 ],
 "slowest_movers": [
  {
   "product": "Fiama Shower Gel 250ml",
   "units": 0
  },
  {
   "product": "Cerelac Wheat 300g",
   "units": 0
  },
  {
   "product": "Real Orange Juice 1L",
   "units": 2
  },
  {
   "product": "Dabur Honey 250g",
   "units": 2
  },
  {
   "product": "Savlon Handwash 200ml",
   "units": 3
  }
 ]
}
--- dead_stock() ---


{
 "count": 3,
 "value_locked_rs": 8910.0,
 "items": [
  {
   "product": "Cerelac Wheat 300g",
   "status": "No sales in 28 days",
   "current_stock": 14,
   "sold_last_28_days": 0,
   "stock_value_rs": 3336.0
  },
  {
   "product": "Fiama Shower Gel 250ml",
   "status": "Overstock",
   "current_stock": 18,
   "sold_last_28_days": 1,
   "stock_value_rs": 3083.0
  },
  {
   "product": "Odomos Cream 50g",
   "status": "Overstock",
   "current_stock": 30,
   "sold_last_28_days": 5,
   "stock_value_rs": 2491.0
  }
 ]
}


In [13]:
print("--- Offline fallback (used if Gemini is down) ---")
for q in ["kya khatam hone wala hai", "expiry", "Amul butter"]:
    print(f"\nQ: {q}\n{it.offline_answer(m, sales_l, q)}")

--- Offline fallback (used if Gemini is down) ---



Q: kya khatam hone wala hai
**Items to reorder:**
- Sunfeast Dark Fantasy 75g: Out of stock, 0 left, order 48
- Maggi 2-Minute Noodles 4-pack 280g: Out of stock, 0 left, order 48
- Amul Butter 100g: Critical, 3 left, order 20
- Surf Excel Easy Wash 1kg: Critical, 2 left, order 24
- Parle-G 250g: Critical, 14 left, order 144
- Maggi 2-Minute Noodles 70g: Critical, 22 left, order 192
- Dabur Red Paste 100g: Critical, 4 left, order 24
- Dabur Chyawanprash 500g: Critical, 5 left, order 24
- Amul Gold Full Cream Milk 500ml: Low, 26 left, order 24

Q: expiry
**Expiring in the next 30 days:**
- Britannia White Bread 400g: expires 04 Oct 2026 (1 day), 7 units likely unsold
- Amul Taaza Toned Milk 500ml: expires 05 Oct 2026 (2 days), 0 units likely unsold
- Amul Malai Paneer 200g: expires 05 Oct 2026 (2 days), 3 units likely unsold
- Amul Gold Full Cream Milk 500ml: expires 05 Oct 2026 (2 days), 0 units likely unsold
- Amul Masti Dahi 400g: expires 06 Oct 2026 (3 days), 9 units likely unsold
-

## 7. Edge cases: names the store does not stock

Early testing showed a dangerous pattern: **"Dairy Milk"** (the Cadbury chocolate) matched the Amul milk pouches, and **"Vim liquid"** matched Vim Bar, so the bot could have answered confidently about the wrong product. The matcher now lists any words that match nothing in the store. If there are such words, the result is a **partial** match: the tool tells the bot not to report that stock as the answer and to say the item is probably not stocked.

In [14]:
for q in ["Dairy Milk", "Vim liquid", "Patanjali atta", "Amul cheese spread", "Kurkure",
          "maggi 70g", "Dabur honey 500", "Parle G 250"]:
    kind, hits = it.find_products(m, q)
    print(f"{q!r:22} -> {kind:9} {hits['product'].head(3).tolist()} unmatched={hits.attrs.get('unmatched_words', [])}")
print()
show(it.check_stock(m, "Dairy Milk"))

'Dairy Milk'           -> partial   ['Amul Taaza Toned Milk 500ml', 'Amul Gold Full Cream Milk 500ml'] unmatched=['dairy']


'Vim liquid'           -> partial   ['Vim Bar 300g'] unmatched=['liquid']
'Patanjali atta'       -> partial   ['Aashirvaad Atta 5kg', 'Aashirvaad Atta 10kg'] unmatched=['patanjali']


'Amul cheese spread'   -> partial   ['Amul Cheese Slices 200g'] unmatched=['spread']


'Kurkure'              -> none      ['Dabur Red Paste 100g', 'Dabur Red Paste 200g', 'Knorr Classic Tomato Soup'] unmatched=[]
'maggi 70g'            -> product   ['Maggi 2-Minute Noodles 70g'] unmatched=[]


'Dabur honey 500'      -> product   ['Dabur Honey 500g'] unmatched=[]


'Parle G 250'          -> product   ['Parle-G 250g'] unmatched=[]

{
 "found": false,
 "partial_match": true,
 "query": "Dairy Milk",
 "unmatched_words": [
  "dairy"
 ],
 "message": "No exact match. The word(s) ['dairy'] match nothing in this store, so the store probably does not stock that item. Closest products are listed only as suggestions; do NOT give their stock as the answer.",
 "closest_products": [
  {
   "product": "Amul Taaza Toned Milk 500ml",
   "current_stock": 42
  },
  {
   "product": "Amul Gold Full Cream Milk 500ml",
   "current_stock": 26
  }
 ]
}


## 8. A known limitation: no festival awareness

The reorder formula uses the last 28 days of sales. It cannot know that Navratri and Diwali are a few weeks away, when agarbatti, biscuits, chocolates and dry snacks sell much faster. The suggestion below is mathematically right for normal weeks but will under-order for festive weeks; the owner has to adjust it.

In [15]:
for p in ["Mangaldeep Agarbatti", "KitKat 4 Finger 37g", "Britannia Good Day Cashew 200g"]:
    r = m[m["product"] == p].iloc[0]
    print(f"{p:32} avg {r['avg_daily_sales']:.1f}/day, stock {r['current_stock']:>3}, "
          f"status {r['status']:<9} suggested order {r['suggested_order_qty']}")

Mangaldeep Agarbatti             avg 1.1/day, stock  22, status OK        suggested order 0
KitKat 4 Finger 37g              avg 3.9/day, stock  79, status OK        suggested order 0
Britannia Good Day Cashew 200g   avg 2.6/day, stock  34, status OK        suggested order 0


## 9. Conclusions

1. The data covers 74 products, 10 brands, 16 categories and 7 distributors over 90 days, with realistic weekly and monthly patterns.
2. Every chatbot tool returns correct, explainable numbers without AI; Gemini only decides **which** tool to call and **how** to explain it.
3. Hindi, Hinglish, misspelt and brand-level queries are matched; ambiguous names (milk, atta, chai patti) return options so the bot asks a clarifying question instead of guessing.
4. Names the store does not stock ("Dairy Milk", "Vim liquid") are flagged as partial matches instead of being answered with the wrong product.
5. Known limitation: the reorder logic has no festival calendar, so it will under-order before Navratri and Diwali. A future version should add a festival uplift factor and weight recent days more.